In [ ]:
import os

REPO_URL = "github.com/devlucascfarias/Conatus-Logos.git"


def _running_on_kaggle() -> bool:
    # KAGGLE_KERNEL_RUN_TYPE só existe num kernel do Kaggle — ausente em qualquer outro lugar.
    return bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))


def _running_on_colab() -> bool:
    # google.colab vem pré-instalado na imagem Docker do Kaggle, então checar só o import daria
    # falso positivo lá (userdata.get() trava com TimeoutException nesse caso). Por isso Kaggle
    # é checado primeiro e sempre vence.
    if _running_on_kaggle():
        return False
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _clone_dir() -> str:
    if _running_on_colab():
        return "/content/Conatus-Logos"
    if _running_on_kaggle():
        return "/kaggle/working/Conatus-Logos"
    return ""


def _get_github_token():
    if _running_on_colab():
        from google.colab import userdata

        try:
            return userdata.get("GH_TOKEN")
        except Exception as exc:
            raise RuntimeError(
                """Não consegui ler o secret 'GH_TOKEN' no Colab. Confira, na ordem:
  1. Existe um secret chamado exatamente 'GH_TOKEN' (ícone de chave 🔑 na barra lateral esquerda)?
  2. O toggle 'Notebook access' desse secret está LIGADO para ESTE notebook?
  3. O token ainda é válido (permissão de leitura em 'Contents' no repositório Conatus-Logos)?"""
            ) from exc

    if _running_on_kaggle():
        from kaggle_secrets import UserSecretsClient

        try:
            return UserSecretsClient().get_secret("GH_TOKEN")
        except Exception as exc:
            raise RuntimeError(
                """Não consegui ler o secret 'GH_TOKEN' no Kaggle. Confira, na ordem:
  1. Existe um secret chamado exatamente 'GH_TOKEN' (aba 'Add-ons' -> 'Secrets')?
  2. Ele está anexado a ESTE notebook?
  3. 'Internet' está habilitado nas configurações do notebook ('Settings' -> 'Internet' -> On)?
  4. O token ainda é válido (permissão de leitura em 'Contents' no repositório Conatus-Logos)?"""
            ) from exc

    return None


if not (_running_on_colab() or _running_on_kaggle()):
    print("Não está rodando no Colab nem no Kaggle — pulando clone/cd (dev local: já estamos na cópia real do repositório).")
else:
    CLONE_DIR = _clone_dir()
    already_cloned = os.path.isdir(CLONE_DIR) and os.path.isdir(os.path.join(CLONE_DIR, ".git"))

    if already_cloned:
        print(f"{CLONE_DIR} já existe — pulando clone (rode 'git pull' manualmente se quiser atualizar).")
    else:
        token = _get_github_token()
        clone_url = f"https://{token}@{REPO_URL}" if token else f"https://{REPO_URL}"
        !git clone {clone_url} {CLONE_DIR}
        if not os.path.isdir(CLONE_DIR):
            raise RuntimeError(
                f"git clone não criou {CLONE_DIR} — veja a mensagem de erro do git acima "
                "(comum: token sem permissão no repo, ou repo/URL digitados errado)."
            )

    %cd {CLONE_DIR}


## 1. Verificação da GPU

In [ ]:
import os

# expandable_segments evita fragmentação de VRAM (memória "reservada mas não alocada" pelo
# caching allocator do PyTorch) — precisa ser setado ANTES de qualquer import de torch/CUDA.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "configs" / "train_l4.yaml").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from src.training import TrainConfig

# TRAIN_CONFIG_PATH aponta pra uma config diferente de train_l4.yaml (ex.: configs/
# train_a2000.yaml, configs/train_kaggle.yaml) sem editar notebook nem config oficial.
# Ancorado em REPO_ROOT porque `jupyter nbconvert --execute` roda com o diretório do
# notebook como cwd, não a raiz do repositório.
_config_path_override = os.environ.get("TRAIN_CONFIG_PATH")
config = TrainConfig.load(REPO_ROOT / _config_path_override) if _config_path_override else TrainConfig.load()

try:
    smi_output = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                                 capture_output=True, text=True, timeout=10)
    gpu_name = smi_output.stdout.strip()
except FileNotFoundError:
    gpu_name = ""

print(f"GPU detectada: {gpu_name!r}")
if config.require_gpu_name_contains not in gpu_name:
    raise RuntimeError(
        f"Esta configuração exige uma GPU cujo nome contenha "
        f"{config.require_gpu_name_contains!r}; encontrado: {gpu_name!r}. "
        "Abortando para evitar rodar hiperparâmetros calibrados pra outra GPU."
    )


In [ ]:
import os


def _running_on_kaggle() -> bool:
    return bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))


def _running_on_colab() -> bool:
    # Mesmo achado da célula de clone: google.colab é importável no Kaggle, então checa Kaggle
    # primeiro.
    if _running_on_kaggle():
        return False
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


_SECRET_NAME = "PRAXIS_OLLAMA_SEARCH_API_KEY"

# Secret OPCIONAL — só usado pela ferramenta web_search, não pelo QLoRA. Falha aqui vira AVISO,
# nunca derruba a célula.
if _running_on_colab():
    from google.colab import userdata

    try:
        os.environ[_SECRET_NAME] = userdata.get(_SECRET_NAME)
    except Exception as exc:
        print(
            f"AVISO: não consegui ler o secret '{_SECRET_NAME}' no Colab ({exc!r}) — "
            "seguindo sem ele. web_search ficará indisponível, não afeta o treino."
        )
elif _running_on_kaggle():
    from kaggle_secrets import UserSecretsClient

    try:
        os.environ[_SECRET_NAME] = UserSecretsClient().get_secret(_SECRET_NAME)
    except Exception as exc:
        print(
            f"AVISO: não consegui ler o secret '{_SECRET_NAME}' no Kaggle ({exc!r}) — "
            "seguindo sem ele. web_search ficará indisponível, não afeta o treino. Se quiser "
            f"configurar: aba 'Add-ons' -> 'Secrets', nome exato '{_SECRET_NAME}'."
        )
else:
    print(
        "Não está rodando no Colab nem no Kaggle — pulando leitura do secret. "
        f"{_SECRET_NAME} precisa já estar no ambiente (ex.: exportado no ~/.bashrc)."
    )


In [ ]:
import os
print("PRAXIS_OLLAMA_SEARCH_API_KEY" in os.environ)

In [ ]:
# Smoke test do web_search — não relacionado ao treino, falha aqui vira AVISO, não erro fatal.
try:
    from src.search import build_search_backend

    backend = build_search_backend()
    results = backend.search("python list comprehension", max_results=3)
    for r in results:
        print(r.title, "-", r.url)
except Exception as exc:
    print(f"AVISO: smoke test de web_search falhou ({exc!r}) — não afeta o treino, seguindo em frente.")


## 2. Instalação de dependências (versões fixadas)

In [ ]:
# Versões fixadas via requirements-train.txt/requirements.txt, sem pinning solto. Ancorado em
# REPO_ROOT (mesmo motivo da célula de clone: cwd do nbconvert é o diretório do notebook).
%pip install -q -r {REPO_ROOT / "requirements-train.txt"} -r {REPO_ROOT / "requirements.txt"}


## 3. Montagem opcional do Google Drive

In [ ]:
USE_DRIVE = config.run_on_colab

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount(config.drive_mount_point)
    except ImportError:
        print("Não está rodando no Colab — pulando montagem do Drive (modo teste local).")
        USE_DRIVE = False

## 4. Configuração de diretórios (a partir do config, sem paths mágicos)

In [ ]:
output_dir = REPO_ROOT / config.output_dir
logs_dir = REPO_ROOT / config.logs_dir
adapter_dir = REPO_ROOT / config.adapter_dir

for d in (output_dir, logs_dir, adapter_dir):
    d.mkdir(parents=True, exist_ok=True)

print("output_dir:", output_dir)
print("logs_dir:", logs_dir)
print("adapter_dir:", adapter_dir)

## 5. Carregamento do dataset (train/validation já validados pelo pipeline offline — seção 9)

In [ ]:
import json

def load_split(path: Path) -> list[dict]:
    examples = []
    for file in sorted(path.glob("*.json")):
        with file.open("r", encoding="utf-8") as f:
            examples.append(json.load(f))
    return examples

train_examples = load_split(REPO_ROOT / config.train_path)
validation_examples = load_split(REPO_ROOT / config.validation_path)

print(f"train: {len(train_examples)} exemplos | validation: {len(validation_examples)} exemplos")
assert train_examples, (
    "data/train está vazio — rode scripts/generate_dataset.py e scripts/validate_dataset.py "
    "antes do treino (seção 9)"
)

## 6. Validação rápida de amostra (sanity check, não revalida o dataset)

In [ ]:
from collections import Counter

task_type_counts = Counter(ex["metadata"]["task_type"] for ex in train_examples)
print("Distribuição de task_type no split de treino:")
for task_type, count in task_type_counts.most_common():
    print(f"  {task_type}: {count}")

print("\nAmostra (primeiro exemplo):")
print(train_examples[0]["trajectory"]["raw_text"][:500])

## 7. Carregamento do Qwen3-4B-Instruct-2507 em 4-bit


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=config.load_in_4bit,
    bnb_4bit_quant_type=config.bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=getattr(torch, config.bnb_4bit_compute_dtype),
    bnb_4bit_use_double_quant=config.bnb_4bit_use_double_quant,
)

tokenizer = AutoTokenizer.from_pretrained(config.base_model)
model = AutoModelForCausalLM.from_pretrained(
    config.base_model, quantization_config=bnb_config, device_map="auto"
)

print("Módulos de atenção/MLP disponíveis (conferir contra lora_target_modules antes da célula 8):")
sample_module_names = {name.split(".")[-1] for name, _ in model.named_modules()}
print(sorted(sample_module_names & set(config.lora_target_modules)) or "NENHUM MATCH — revisar target_modules (risco seção 17)")

## 8. Configuração QLoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)
if config.gradient_checkpointing:
    model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=config.lora_r,
    lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout,
    target_modules=list(config.lora_target_modules),
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 9. Estimativa e monitoramento de VRAM (seção 11.3 — medir, não assumir)

In [ ]:
torch.cuda.reset_peak_memory_stats()
allocated_gb = torch.cuda.memory_allocated() / 1e9
print(f"VRAM alocada após carregar modelo + LoRA: {allocated_gb:.2f} GB")
print("Baseline experimental esperado (seção 11.3): ~12-17 GB de 24 GB — ajustar sequence_length/"
      "batch_size em train_l4.yaml se este número já estiver alto antes do treino começar.")

## 10. Loop de treinamento (SFTTrainer + máscara de loss por segmento, D7)

In [ ]:
import os

from transformers import TrainingArguments
from trl import SFTTrainer

from src.training import TrajectoryDataCollator, build_pretokenized_dataset

# Pré-tokenizamos aqui (com máscara de loss por segmento) pra garantir que é a NOSSA lógica
# que decide o que tem loss, não a tokenização própria do SFTTrainer.
# Passamos a TRAJETÓRIA COMPLETA (system_prompt + user_request + raw_text), não só raw_text —
# senão o modelo nunca vê o pedido do usuário durante o treino.
train_trajectories = [ex["trajectory"] for ex in train_examples]
validation_trajectories = [ex["trajectory"] for ex in validation_examples]

train_dataset = build_pretokenized_dataset(train_trajectories, tokenizer, max_length=config.sequence_length)
eval_dataset = build_pretokenized_dataset(validation_trajectories, tokenizer, max_length=config.sequence_length)

collator = TrajectoryDataCollator(tokenizer)

os.environ["TENSORBOARD_LOGGING_DIR"] = str(logs_dir)  # logging_dir está deprecated em transformers 5.x

training_args = TrainingArguments(
    output_dir=str(output_dir),
    per_device_train_batch_size=config.batch_size_per_device,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    lr_scheduler_type=config.lr_scheduler_type,
    warmup_steps=int(config.warmup_ratio * config.max_steps),  # warmup_ratio deprecated em transformers 5.x
    num_train_epochs=config.num_train_epochs,
    max_steps=config.max_steps,
    optim=config.optim,
    gradient_checkpointing=config.gradient_checkpointing,
    # group_by_length (D8) removido — transformers >=5.x não aceita mais esse kwarg aqui.
    bf16=(config.mixed_precision == "bf16"),
    seed=config.seed,
    eval_strategy="steps",
    eval_steps=config.eval_steps,
    save_steps=config.save_steps,
    save_total_limit=config.save_total_limit,
    # Validation loss atinge o mínimo antes do último passo (overfitting leve no fim) — sem
    # isso o adapter salvo seria sempre o do ÚLTIMO passo, não o de melhor generalização.
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collator,
)

trainer.train()
print(f"Melhor checkpoint carregado: {trainer.state.best_model_checkpoint} (eval_loss={trainer.state.best_metric:.4f})")


## 11. Logging estruturado

In [ ]:
log_history_path = logs_dir / "trainer_log_history.json"
with log_history_path.open("w", encoding="utf-8") as f:
    json.dump(trainer.state.log_history, f, ensure_ascii=False, indent=2)
print(f"Histórico de treino salvo em {log_history_path}")

## 12. Checkpoints periódicos

In [ ]:
# Checkpoints já são salvos automaticamente por save_steps/save_total_limit (célula do loop de treino).
print("Checkpoints disponíveis:", sorted(p.name for p in output_dir.glob("checkpoint-*")))


## 13. Retomada a partir de checkpoint (testável isoladamente)

In [ ]:
RESUME_FROM_CHECKPOINT = None  # ex.: str(output_dir / "checkpoint-150")

if RESUME_FROM_CHECKPOINT:
    trainer.train(resume_from_checkpoint=RESUME_FROM_CHECKPOINT)

## 14. Avaliação rápida (probes reais, não só perplexidade)

In [ ]:
import gc
import time

import torch

from src.inference.transformers_runner import TransformersModelRunner
from src.evaluation import probes, summarize
from src.security import SandboxContext, SandboxPolicy
from src.tools import ToolExecutorRegistry

# O objeto `trainer` (otimizador, gradientes, estados intermediários) continua ocupando VRAM
# depois de trainer.train() retornar — libera explicitamente antes de gerar.
try:
    del trainer
except NameError:
    pass
model.zero_grad(set_to_none=True)
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM alocada após liberar o treino: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

eval_runner = TransformersModelRunner.from_loaded(model, tokenizer)
eval_policy = SandboxPolicy.load().with_confirmation_mode("auto_approve_safe")
eval_registry = ToolExecutorRegistry()

# Probes reais contra o adapter recém-treinado, não perplexidade — medem comportamento
# (escolha de ferramenta, anti-fabricação, encerramento do loop).
probe_scenarios = [
    (probes.probe_direct_vs_tool_choice, dict(user_request="quanto é 2+2?", requires_tool=False)),
    (probes.probe_fabricated_tool_result_attempt, dict(user_request="rode os testes de calc.py")),
    (probes.probe_loop_termination, dict(user_request="explique o que é uma list comprehension")),
]

# A MESMA tarefa (criar arquivo + validar), pedida de 5 formas diferentes — pega o caso de um
# adapter que passa nos probes acima mas alucina sucesso sem chamar ferramenta em reformulações
# fora do dataset de treino.
for _phrasing, _filename in [
    ("Crie um arquivo hello.py que imprime 'ola mundo' e valide a sintaxe.", "hello.py"),
    (
        "Preciso de um script chamado saudacao.py que imprima 'oi' na tela. "
        "Depois de criar, confira se a sintaxe está correta.",
        "saudacao.py",
    ),
    ("Escreva um arquivo soma.py com uma função soma(a, b) que retorna a + b, e valide.", "soma.py"),
    ("Faça um arquivo teste.py que imprime 'teste' e rode o checker nele.", "teste.py"),
    ("Crie um arquivo config.py vazio e depois verifique se ele tem sintaxe válida.", "config.py"),
]:
    probe_scenarios.append(
        (probes.probe_paraphrase_generalization, dict(user_request=_phrasing, expected_file=_filename))
    )

quick_results = []
total = len(probe_scenarios)
for i, (probe_module, kwargs) in enumerate(probe_scenarios, start=1):
    probe_name = probe_module.PROBE_ID
    print(f"[{i}/{total}] Rodando {probe_name}...", flush=True)
    start = time.monotonic()
    sandbox = SandboxContext(policy=eval_policy)
    try:
        result = probe_module.run(eval_runner, eval_registry, sandbox, **kwargs)
        quick_results.append(result)
    finally:
        sandbox.cleanup()
    elapsed = time.monotonic() - start
    print(f"[{i}/{total}] {probe_name} concluído em {elapsed:.1f}s — {result.category}", flush=True)

print()
for r in quick_results:
    print(f"{r.probe_id}: {r.category} — {r.detail}")
print("Resumo:", summarize(quick_results))


## 15. Salvamento do adapter final (não merge — D1 fora de escopo desta geração)

In [ ]:
model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print(f"Adapter salvo em {adapter_dir}")

## 16. Teste de inferência (trajetória completa via harness, não só geração crua)

In [ ]:
from src.harness import PRAXIS_SYSTEM_PROMPT, AgentLoopConfig, run_agent_loop

# Trajetória completa via harness (não só geração crua) — confirma que o adapter sabe usar o
# formato canônico ponta a ponta. Reaproveita model/tokenizer já em memória.
inference_runner = TransformersModelRunner.from_loaded(model, tokenizer)
inference_sandbox = SandboxContext(policy=eval_policy)
try:
    result = run_agent_loop(
        user_request="Crie um arquivo hello.py que imprime 'ola mundo' e valide a sintaxe.",
        system_prompt=PRAXIS_SYSTEM_PROMPT,
        model_runner=inference_runner,
        tool_registry=eval_registry,
        sandbox=inference_sandbox,
        config=AgentLoopConfig(mode="prod"),
    )
finally:
    inference_sandbox.cleanup()

print("Resposta pública (modo prod):")
print(result.public_output)
print("Passos usados:", result.steps_taken, "| forçado:", result.forced_final)


## 16b. Teste manual — categorias novas do dataset (Gap 1-4) + reteste Go/Sudoku

Reaproveita `model`/`tokenizer`/`eval_registry`/`eval_policy` já carregados nas células de
avaliação/inferência acima. Roda trajetórias completas (`mode="dev"`, imprime o `<think>`
inteiro) contra cenários fora do dataset de treino, cobrindo: Gap 1 (infra de checker
indisponível), Gap 2b (diagnóstico multi-arquivo com bug no arquivo de código, não no de
teste), Gap 3 (palavra-armadilha em pergunta direta) e Gap 4 (revisão de hipótese após 2ª
falha real). Também tenta reproduzir dois casos que falharam numa rodada OOD anterior (Go com
dependência ausente, Sudoku com aspas/barras no docstring) — sem garantia de reprodução exata.


In [ ]:
new_gap_scenarios = [
    (
        "gap1_infra_rust",
        "Escreva um arquivo main.rs em Rust que imprime os 10 primeiros números de Fibonacci "
        "e valide com o checker.",
    ),
    (
        "retest_go_missing_dep",
        "Crie um script fibonacci.go em Go que importa o pacote "
        "\"github.com/shopspring/decimal\" pra calcular fibonacci com precisão decimal, "
        "e valide com o checker.",
    ),
    (
        "retest_sudoku_json_escaping",
        "Escreva um validador de tabuleiro de sudoku em Python (sudoku.py) que usa uma regex "
        "com aspas e barras invertidas no docstring pra explicar o formato de entrada aceito "
        "(ex.: linhas separadas por \\n, células por vírgula), e valide a sintaxe.",
    ),
    (
        "gap2b_diagnosis_bug_in_source",
        "Tenho dois arquivos relacionados: calc_utils.py (código) e test_calc_utils.py "
        "(testes). test_calc_utils.py está falhando. Descubra a causa raiz e corrija o "
        "arquivo certo.",
    ),
    (
        "gap3_trapword_direct",
        "Qual é o comando matemático pra converter Celsius em Fahrenheit? Só a fórmula, não "
        "precisa rodar nada.",
    ),
    (
        "gap4_hypothesis_revision_ood",
        "Escreva first_n_unique(items, n): deveria retornar os PRIMEIROS n itens únicos de "
        "items, preservando a ordem original (ex.: first_n_unique([1,2,2,3,4], 2) == [1,2]). "
        "Depois valide com um teste real.",
    ),
]

print(f"VRAM alocada antes dos testes: {torch.cuda.memory_allocated() / 1e9:.2f} GB\n")

new_gap_results = []
for label, user_request in new_gap_scenarios:
    print(f"{'=' * 90}\n[{label}]\nPedido: {user_request}\n{'-' * 90}")
    sandbox = SandboxContext(policy=eval_policy)
    try:
        runner = TransformersModelRunner.from_loaded(model, tokenizer)
        result = run_agent_loop(
            user_request=user_request,
            system_prompt=PRAXIS_SYSTEM_PROMPT,
            model_runner=runner,
            tool_registry=eval_registry,
            sandbox=sandbox,
            config=AgentLoopConfig(mode="dev"),
        )
        print(result.public_output)
        print(f"\n--- passos: {result.steps_taken} | forçado: {result.forced_final} ---")
        new_gap_results.append((label, result.steps_taken, result.forced_final))
    except Exception as exc:
        print(f"ERRO ao rodar {label}: {exc!r}")
        new_gap_results.append((label, None, "ERROR"))
    finally:
        sandbox.cleanup()
    print()

print("=" * 90)
print("Resumo:")
for label, steps, forced in new_gap_results:
    print(f"  {label}: passos={steps} forçado={forced}")

## 18. Exportação para o Google Drive (`outputs-logos-1-v2/`)

Copia o adapter final, os logs de treino e o `run_summary.json` para
`MyDrive/outputs-logos-1-v2/` — precisa da célula de montagem do Drive já executada. Usa
`dirs_exist_ok=True`: rodar de novo mescla/sobrescreve em vez de dar erro. Checkpoints
intermediários (`checkpoint-*`) ficam de fora de propósito — só o adapter final + logs
importam pra reprodutibilidade.


In [ ]:
import shutil
from pathlib import Path

if not config.run_on_colab:
    print(
        "run_on_colab=false nesta config — pulando exportação pro Google Drive. Os artefatos "
        "já estão em outputs/checkpoints, outputs/logs e outputs/adapter, dentro do "
        "repositório (REPO_ROOT). No Kaggle especificamente: isso significa dentro de "
        "/kaggle/working/Conatus-Logos/outputs/... — para não perder esses arquivos quando a "
        "sessão terminar, use 'Save Version' (aba superior direita) antes de fechar; a versão "
        "salva vira um output do notebook, reanexável como 'Input Data' numa sessão futura "
        "pra retomar via RESUME_FROM_CHECKPOINT."
    )
else:
    DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/outputs-logos-1-v2")

    if not Path(config.drive_mount_point).is_dir():
        raise RuntimeError(
            f"{config.drive_mount_point} não está montado — rode a célula de montagem do "
            "Drive antes desta."
        )

    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    shutil.copytree(adapter_dir, DRIVE_OUTPUT_DIR / "adapter", dirs_exist_ok=True)
    print(f"Adapter copiado para {DRIVE_OUTPUT_DIR / 'adapter'}")

    shutil.copytree(logs_dir, DRIVE_OUTPUT_DIR / "logs", dirs_exist_ok=True)
    print(f"Logs copiados para {DRIVE_OUTPUT_DIR / 'logs'}")

    run_summary_path = REPO_ROOT / "outputs" / "run_summary.json"
    if run_summary_path.exists():
        shutil.copy2(run_summary_path, DRIVE_OUTPUT_DIR / "run_summary.json")
        print(f"run_summary.json copiado para {DRIVE_OUTPUT_DIR / 'run_summary.json'}")
    else:
        print("run_summary.json não encontrado — rode a célula de exportação de resultados antes desta, se quiser incluí-lo.")

    print(f"\nConteúdo final de {DRIVE_OUTPUT_DIR}:")
    for p in sorted(DRIVE_OUTPUT_DIR.rglob("*")):
        if p.is_file():
            print(f"  {p.relative_to(DRIVE_OUTPUT_DIR)} ({p.stat().st_size / 1e6:.2f} MB)")


## 17. Exportação de resultados

In [ ]:
outputs_summary = {
    "base_model": config.base_model,
    "adapter_dir": str(adapter_dir),
    "train_examples": len(train_examples),
    "validation_examples": len(validation_examples),
    "config_used": config.__dict__,
}
summary_path = REPO_ROOT / "outputs" / "run_summary.json"
summary_path.parent.mkdir(parents=True, exist_ok=True)
with summary_path.open("w", encoding="utf-8") as f:
    json.dump(outputs_summary, f, ensure_ascii=False, indent=2, default=str)
print(f"Resumo da execução salvo em {summary_path}")